# Optional Project - Colab Part2 Training

This notebook runs Part 2: Tiny LR sweep, five standard-parameterization model sizes, checkpoint sync to Google Drive, and scaling-law fit. Re-run interrupted training cells to resume from Drive checkpoints.

In [ ]:
# ===== User config =====
REPO_URL = "https://github.com/Peng-y-x/optionalproject.git"
REPO_DIR = "/content/optionalproject"
REPO_BRANCH = "run"
SWEEP_CONFIG = "configs/sweep_lr.yaml"
BEST_LR_JSON = "outputs/part2_lr_sweep/best_lr.json"
DRIVE_BEST_LR_JSON = "/content/drive/MyDrive/svg-scaling/part2_lr_sweep/best_lr.json"

In [ ]:
# 1) Clone repo and checkout branch
import os
if not os.path.exists(REPO_DIR):
    !git clone --branch {REPO_BRANCH} --single-branch {REPO_URL} {REPO_DIR}
else:
    print('Repo already exists:', REPO_DIR)
%cd $REPO_DIR
!git fetch origin
!git checkout {REPO_BRANCH}
!git pull origin {REPO_BRANCH}
!git branch --show-current
!git rev-parse --short HEAD

In [ ]:
# 2) Mount Google Drive for resumable checkpoints
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/svg-scaling/part2
!mkdir -p /content/drive/MyDrive/svg-scaling/part2_lr_sweep

In [ ]:
# 3) Install system + Python dependencies
!apt-get update -y
!apt-get install -y libcairo2 libcairo2-dev libffi-dev
!python -m pip install --upgrade pip
!pip install -r requirements.txt

In [ ]:
# 4) HF auth from Colab Keys (key name must be HF_TOKEN)
import os
from google.colab import userdata
token = userdata.get('HF_TOKEN')
if token:
    os.environ['HF_TOKEN'] = token
    print('HF token loaded from Colab key.')
else:
    print('HF token not found in Colab key HF_TOKEN. Public dataset loading may still work.')
print('has_hf_token:', bool(os.getenv('HF_TOKEN')))

In [ ]:
# 5) GPU sanity check
import torch
print('torch', torch.__version__)
print('cuda_available', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu', torch.cuda.get_device_name(0))
    print('bf16_supported', torch.cuda.is_bf16_supported())

In [ ]:
# 6) Part 2 LR sweep on Tiny model
# If Colab disconnects, rerun this cell; each LR run resumes from Drive latest.pt.
%cd $REPO_DIR
!python scripts/run_lr_sweep.py --config {SWEEP_CONFIG}

In [ ]:
# 7) Inspect best LR
import json, os, shutil
from pathlib import Path
if not Path(BEST_LR_JSON).exists() and Path(DRIVE_BEST_LR_JSON).exists():
    Path(BEST_LR_JSON).parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(DRIVE_BEST_LR_JSON, BEST_LR_JSON)
best = json.loads(Path(BEST_LR_JSON).read_text())
print(json.dumps(best, indent=2))
BEST_LR = best['learning_rate']
print('BEST_LR=', BEST_LR)

In [ ]:
# 8) Train all five model sizes for exactly one epoch with the selected fixed LR
# If Colab disconnects, rerun this cell; each model resumes from Drive latest.pt.
%cd $REPO_DIR
!python scripts/run_part2_all.py --best-lr-json {BEST_LR_JSON}

In [ ]:
# 9) Fit Part 2 scaling law and create plot/table
%cd $REPO_DIR
!python scripts/fit_scaling_law.py --runs-dir outputs/part2 --output-dir outputs/part2_analysis
!mkdir -p /content/drive/MyDrive/svg-scaling/part2_analysis
!cp -r outputs/part2_analysis/* /content/drive/MyDrive/svg-scaling/part2_analysis/

In [ ]:
# 10) Inspect key Part 2 outputs
from pathlib import Path
import json
for p in sorted(Path('outputs/part2').glob('*/final_metrics.json')):
    m=json.loads(p.read_text())
    print(p.parent.name, {k:m.get(k) for k in ['num_parameters','val_loss','val_ppl','tokens_seen','wall_clock_seconds','peak_gpu_memory_gb']})
fit_path=Path('outputs/part2_analysis/part2_scaling_fit.json')
if fit_path.exists():
    print(json.dumps(json.loads(fit_path.read_text())['fit'], indent=2))